# Data Engineering Interview Prep: Easy SQL (Complete 1-21)
## Topic: Filtering, Joins, Aggregations, and Date Math

---

### **Easy Lesson 1: Twitter - "Histogram of Tweets"**

#### **The Problem**
Write a query to obtain a histogram of tweets posted per user in 2022. Output the tweet bucket and the number of users in that bucket.

#### **The Logic**
Use a two-step aggregation: first, count the tweets per user in 2022. Then, group by those counts to find how many users fall into each bucket.

#### **The Solution (PostgreSQL)**
```sql
WITH tweet_counts AS (
  SELECT user_id, COUNT(tweet_id) AS tweet_bucket
  FROM tweets
  WHERE tweet_date >= '2022-01-01' AND tweet_date < '2023-01-01'
  GROUP BY user_id
)
SELECT tweet_bucket, COUNT(user_id) AS users_num
FROM tweet_counts
GROUP BY tweet_bucket
ORDER BY tweet_bucket;
```

#### **Senior Data Engineer Perspective**
Ensure massive tables are partitioned by `tweet_date` for partition pruning. Be aware of data skew during the first `GROUP BY` if "Power Users" have significantly more tweets than standard users.

---

### **Easy Lesson 2: LinkedIn - "Data Science Skills"**

#### **The Problem**
List candidate IDs who possess Python, Tableau, and PostgreSQL. Order ascending.

#### **The Logic**
Filter for the required skills, group by the candidate, and use `HAVING` to ensure they possess exactly 3 distinct required skills.

#### **The Solution (PostgreSQL)**
```sql
SELECT candidate_id
FROM candidates
WHERE skill IN ('Python', 'Tableau', 'PostgreSQL')
GROUP BY candidate_id
HAVING COUNT(skill) = 3
ORDER BY candidate_id ASC;
```

#### **Senior Data Engineer Perspective**
The `IN` clause enables predicate pushdown, minimizing data sent to the shuffle phase. Using `COUNT(DISTINCT skill)` is safer to protect against duplicate skill entries in messy source data.

---

### **Easy Lesson 3: Facebook - "Page With No Likes"**

#### **The Problem**
Return the IDs of Facebook pages that have zero likes.

#### **The Logic**
Perform a `LEFT JOIN` from pages to likes and filter for where the likes side `IS NULL`.

#### **The Solution (PostgreSQL)**
```sql
SELECT p.page_id
FROM pages p
LEFT JOIN page_likes pl ON p.page_id = pl.page_id
WHERE pl.page_id IS NULL
ORDER BY p.page_id ASC;
```

#### **Senior Data Engineer Perspective**
Prefer `LEFT JOIN ... IS NULL` over `NOT IN`, as `NOT IN` fails if the subquery returns any `NULL` values. In PySpark, this is executed efficiently as a `left_anti` join.

---

### **Easy Lesson 4: Tesla - "Unfinished Parts"**

#### **The Problem**
Find parts that have begun assembly but lack a finish date.

#### **The Logic**
Filter the table for rows where the `finish_date` is missing.

#### **The Solution (PostgreSQL)**
```sql
SELECT part, assembly_step
FROM parts_assembly
WHERE finish_date IS NULL;
```

#### **Senior Data Engineer Perspective**
Columnar formats like Parquet track null counts in their metadata, making `IS NULL` checks fast. If queried frequently, partitioning by an `is_finished` boolean flag is better than scanning for nulls.

---

### **Easy Lesson 5: NY Times - "Laptop vs. Mobile Viewership"**

#### **The Problem**
Calculate total viewership for laptops vs. mobile (tablet + phone).

#### **The Logic**
Use conditional aggregation (`SUM(CASE WHEN...)`) to route the counts in a single pass.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  SUM(CASE WHEN device_type = 'laptop' THEN 1 ELSE 0 END) AS laptop_views,
  SUM(CASE WHEN device_type IN ('tablet', 'phone') THEN 1 ELSE 0 END) AS mobile_views
FROM viewership;
```

#### **Senior Data Engineer Perspective**
Conditional aggregation scans the table exactly once. In Spark, `F.sum(F.when(...))` is more performant than using `groupBy().pivot()` because it avoids dynamic pivoting overhead.

---

### **Easy Lesson 6: Facebook - "Average Post Hiatus (Part 1)"**

#### **The Problem**
Find the days between a user's first and last post in 2021 for users with at least 2 posts.

#### **The Logic**
Group by user, apply a `HAVING` clause for the count, and subtract the min date from the max date.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  user_id, 
  EXTRACT(DAY FROM MAX(post_date) - MIN(post_date)) AS days_between
FROM posts
WHERE post_date >= '2021-01-01' AND post_date < '2022-01-01'
GROUP BY user_id
HAVING COUNT(post_id) >= 2;
```

#### **Senior Data Engineer Perspective**
Applying functions to columns in the `WHERE` clause (like `EXTRACT(YEAR FROM date)`) ruins index usage. Always use raw date bounds for sargable queries.

---

### **Easy Lesson 7: Microsoft - "Teams Power Users"**

#### **The Problem**
Identify the top 2 users who sent the most messages in August 2022.

#### **The Logic**
Group by sender, count messages, order descending, and limit to 2.

#### **The Solution (PostgreSQL)**
```sql
SELECT sender_id, COUNT(message_id) AS message_count
FROM messages
WHERE sent_date >= '2022-08-01' AND sent_date < '2022-09-01'
GROUP BY sender_id
ORDER BY message_count DESC
LIMIT 2;
```

#### **Senior Data Engineer Perspective**
Global `ORDER BY` in distributed systems pushes all data to a single node. Query engines usually optimize small limits using local "Top-K" algorithms before merging.

---

### **Easy Lesson 8: LinkedIn - "Duplicate Job Listings"**

#### **The Problem**
Count companies that posted multiple jobs with the exact same title and description.

#### **The Logic**
Group by company, title, and description, filter for counts > 1, then count the distinct companies from that result.

#### **The Solution (PostgreSQL)**
```sql
WITH duplicates AS (
  SELECT company_id
  FROM job_listings
  GROUP BY company_id, title, description
  HAVING COUNT(job_id) > 1
)
SELECT COUNT(DISTINCT company_id) AS duplicate_companies
FROM duplicates;
```

#### **Senior Data Engineer Perspective**
Grouping by massive text blocks destroys memory. Generate an MD5/SHA-256 hash of the text during ingestion and group by the hash instead.

---

### **Easy Lesson 9: Robinhood - "Cities With Completed Trades"**

#### **The Problem**
Retrieve the top three cities with the most 'Completed' trades.

#### **The Logic**
Join trades to users, filter for 'Completed', group by city, sort, and limit.

#### **The Solution (PostgreSQL)**
```sql
SELECT u.city, COUNT(t.order_id) AS total_orders
FROM trades t
JOIN users u ON t.user_id = u.user_id
WHERE t.status = 'Completed'
GROUP BY u.city
ORDER BY total_orders DESC
LIMIT 3;
```

#### **Senior Data Engineer Perspective**
This is a classic Fact-to-Dimension join. Ensure the smaller `users` table is broadcasted across worker nodes (Broadcast Hash Join) to prevent shuffling the massive `trades` table.

---

### **Easy Lesson 10: Amazon - "Average Review Ratings"**

#### **The Problem**
Get the average star rating per product, grouped by month.

#### **The Logic**
Extract the month, group by month and product, and round the average.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  EXTRACT(MONTH FROM submit_date) AS mth,
  product_id,
  ROUND(AVG(stars), 2) AS avg_stars
FROM reviews
GROUP BY EXTRACT(MONTH FROM submit_date), product_id
ORDER BY mth, product_id;
```

#### **Senior Data Engineer Perspective**
Do not run heavy aggregations directly against source tables for BI dashboards. Schedule an Airflow/Spark job to calculate these and write to a Materialized View.

---

### **Easy Lesson 11: FAANG - "Well Paid Employees"**

#### **The Problem**
Identify employees who earn more than their direct managers.

#### **The Logic**
Perform a self-join comparing the employee's manager_id to the manager's employee_id.

#### **The Solution (PostgreSQL)**
```sql
SELECT e.name AS employee_name
FROM employee e
JOIN employee m ON e.manager_id = m.employee_id
WHERE e.salary > m.salary;
```

#### **Senior Data Engineer Perspective**
Self-joins on large hierarchy tables trigger massive network shuffles. Flatten hierarchies into arrays or use Spark GraphFrames for large-scale organizational data.

---

### **Easy Lesson 12: PayPal - "Final Account Balance"**

#### **The Problem**
Calculate final balance from a ledger of deposits and withdrawals.

#### **The Logic**
Use conditional math inside a `SUM()` to add deposits and subtract withdrawals.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  account_id,
  SUM(CASE 
      WHEN transaction_type = 'Deposit' THEN amount 
      WHEN transaction_type = 'Withdrawal' THEN -amount 
      ELSE 0 END) AS final_balance
FROM transactions
GROUP BY account_id;
```

#### **Senior Data Engineer Perspective**
Financial pipelines must be strictly idempotent. Ensure the architecture relies on processed-event tracking to prevent double-counting if a worker node crashes mid-execution.

---

### **Easy Lesson 13: Facebook - "App Click-through Rate (CTR)"**

#### **The Problem**
Calculate CTR (Clicks / Impressions) for apps in 2022.

#### **The Logic**
Use conditional sums to isolate clicks and impressions, then divide.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  app_id,
  ROUND(100.0 * SUM(CASE WHEN event_type = 'click' THEN 1 ELSE 0 END) / 
    NULLIF(SUM(CASE WHEN event_type = 'impression' THEN 1 ELSE 0 END), 0), 2) AS ctr
FROM events
WHERE timestamp >= '2022-01-01' AND timestamp < '2023-01-01'
GROUP BY app_id;
```

#### **Senior Data Engineer Perspective**
`NULLIF(val, 0)` is mandatory. Without it, anomalous data where impressions equal zero will cause a fatal Division by Zero error.

---

### **Easy Lesson 14: TikTok - "Second Day Confirmation"**

#### **The Problem**
Find users who confirmed sign-up exactly one day after initiating it.

#### **The Logic**
Join the tables and filter where action date equals signup date plus a 1-day interval.

#### **The Solution (PostgreSQL)**
```sql
SELECT e.user_id
FROM emails e
JOIN texts t ON e.email_id = t.email_id
WHERE t.signup_action = 'Confirmed'
  AND t.action_date = e.signup_date + INTERVAL '1 day';
```

#### **Senior Data Engineer Perspective**
Standardize all ingestion to UTC. When building date logic, differentiate between strict 24-hour windows and calendar midnight crossings.

---

### **Easy Lesson 15: IBM - "IBM db2 Product Analytics"**

#### **The Problem**
Histogram of unique queries per employee in Q3, including 0-query employees.

#### **The Logic**
`LEFT JOIN` employees to queries, ensuring the date filter is inside the `ON` clause, not `WHERE`.

#### **The Solution (PostgreSQL)**
```sql
WITH query_counts AS (
  SELECT e.emp_id, COUNT(q.query_id) AS query_count
  FROM employees e
  LEFT JOIN queries q 
    ON e.emp_id = q.emp_id 
    AND q.query_date >= '2023-07-01' AND q.query_date < '2023-10-01'
  GROUP BY e.emp_id
)
SELECT query_count AS unique_queries, COUNT(emp_id) AS employee_count
FROM query_counts
GROUP BY query_count
ORDER BY unique_queries;
```

#### **Senior Data Engineer Perspective**
The date filter must live in the `ON` clause. If placed in the `WHERE` clause, null `query_date` rows are filtered out, turning the `LEFT JOIN` into an `INNER JOIN`.

---

### **Easy Lesson 16: JPMorgan - "Cards Issued Difference"**

#### **The Problem**
Find the difference between max and min issued amounts for each card.

#### **The Logic**
Group by card name and subtract `MIN()` from `MAX()`.

#### **The Solution (PostgreSQL)**
```sql
SELECT card_name, MAX(issued_amount) - MIN(issued_amount) AS difference
FROM monthly_cards_issued
GROUP BY card_name
ORDER BY difference DESC;
```

#### **Senior Data Engineer Perspective**
Query engines using Parquet or ORC can execute MIN/MAX aggregations by simply reading file footer metadata, bypassing row-level scanning entirely.

---

### **Easy Lesson 17: Alibaba - "Compressed Mean"**

#### **The Problem**
Find mean items per order given a pre-aggregated table of counts.

#### **The Logic**
Multiply counts by occurrences, sum them, cast to decimal, and divide by total occurrences.

#### **The Solution (PostgreSQL)**
```sql
SELECT ROUND(
    SUM(item_count::DECIMAL * order_occurrences) / SUM(order_occurrences)
  , 1) AS mean
FROM items_per_order;
```

#### **Senior Data Engineer Perspective**
Prevent integer truncation bugs by explicitly casting the numerator to `DECIMAL`. Storing data in this "compressed" format is a standard Data Warehousing pre-aggregation technique.

---

### **Easy Lesson 18: CVS Health - "Pharmacy Analytics (Part 1)"**

#### **The Problem**
Find the top 3 most profitable drugs.

#### **The Logic**
Subtract cogs from sales directly, sort descending, and limit.

#### **The Solution (PostgreSQL)**
```sql
SELECT drug, (total_sales - cogs) AS total_profit
FROM pharmacy_sales
ORDER BY total_profit DESC
LIMIT 3;
```

#### **Senior Data Engineer Perspective**
Row-level math is extremely fast in Spark. Project Tungsten optimizes this using SIMD (Single Instruction, Multiple Data) execution directly in CPU registers.

---

### **Easy Lesson 19: CVS Health - "Pharmacy Analytics (Part 2)"**

#### **The Problem**
Identify manufacturers operating at a loss, showing drug count and total loss.

#### **The Logic**
Filter for losses first, then group by manufacturer and aggregate.

#### **The Solution (PostgreSQL)**
```sql
SELECT manufacturer, COUNT(drug) AS drug_count, SUM(cogs - total_sales) AS total_loss
FROM pharmacy_sales
WHERE cogs > total_sales
GROUP BY manufacturer
ORDER BY total_loss DESC;
```

#### **Senior Data Engineer Perspective**
Predicate pushdown priority is key here. Filtering out profitable drugs *before* the `GROUP BY` drastically reduces memory overhead during the shuffle phase.

---

### **Easy Lesson 20: CVS Health - "Pharmacy Analytics (Part 3)"**

#### **The Problem**
Format total sales into a string like "$36 million".

#### **The Logic**
Sum the sales, divide by 1M, round, and wrap in a `CONCAT()` function.

#### **The Solution (PostgreSQL)**
```sql
SELECT manufacturer, CONCAT('$', ROUND(SUM(total_sales) / 1000000), ' million') AS sale_format
FROM pharmacy_sales
GROUP BY manufacturer
ORDER BY SUM(total_sales) DESC, manufacturer ASC;
```

#### **Senior Data Engineer Perspective**
Separate the data and presentation layers. Provide pure numeric outputs to the BI tool and handle string formatting there to allow for downstream math and sorting.

---

### **Easy Lesson 21: UnitedHealth - "Patient Support Analysis (Part 1)"**

#### **The Problem**
Count total policyholders who called 3 or more times.

#### **The Logic**
Use a CTE to find policyholders with >= 3 calls, then do a global count of those users.

#### **The Solution (PostgreSQL)**
```sql
WITH heavy_callers AS (
  SELECT policy_holder_id
  FROM callers
  GROUP BY policy_holder_id
  HAVING COUNT(case_id) >= 3
)
SELECT COUNT(policy_holder_id) AS member_count
FROM heavy_callers;
```

#### **Senior Data Engineer Perspective**
A `COUNT()` without a `GROUP BY` requires all data to funnel into a single executor node. If the intermediate CTE result is massive, this causes Out-Of-Memory exceptions.

# Data Engineering Interview Prep: Medium SQL (Lessons 1 - 4)
## Topic: Window Functions, Advanced CTEs, and Rolling Averages

---

### **Medium Lesson 1: Uber - "User's Third Transaction"**

#### **The Problem**
Write a query to obtain the third transaction of every user. Output the `user_id`, `spend`, and `transaction_date`.
*(Table: `transactions` [user_id, spend, transaction_date])*

#### **The Logic (Window Functions)**
We need to rank each user's transactions chronologically. We cannot just use a `GROUP BY` because we need the specific row's details. We will use the `ROW_NUMBER()` window function to assign a sequence number to each transaction, partitioned by the user.

#### **The Solution (PostgreSQL)**
```sql
WITH ranked_transactions AS (
  SELECT 
    user_id, 
    spend, 
    transaction_date,
    ROW_NUMBER() OVER (
      PARTITION BY user_id 
      ORDER BY transaction_date
    ) AS transaction_rank
  FROM transactions
)

SELECT 
  user_id, 
  spend, 
  transaction_date
FROM ranked_transactions
WHERE transaction_rank = 3;
```

### Senior Data Engineer Perspective
- The Cost of Window Functions: In PySpark or Databricks, a Window.partitionBy("user_id") requires a massive data shuffle. All transactions for a specific user must be moved across the network to reside on the exact same worker node.

- Data Skew Danger: If you are processing passenger booking events, a standard traveler might have 5 flights a year, but a corporate travel agency ID might have 500,000. That single massive partition can overwhelm a node and cause an Out-Of-Memory (OOM) error. Understanding how to handle skewed partitions (e.g., using salted keys) is a key senior-level design consideration.

## **Medium Lesson 2: FAANG - "Second Highest Salary"**
#### **The Problem**
Write a query to find the second-highest salary from an employee table. If there is no second-highest salary, return NULL.
(Table: employee [employee_id, salary])

### **The Logic (Subqueries vs. Ranking)**
While you could use DENSE_RANK(), a simpler and often more highly optimized approach is to find the maximum salary that is strictly less than the absolute maximum salary.

### **The Solution (PostgreSQL)**
```SQL
SELECT MAX(salary) AS second_highest_salary
FROM employee
WHERE salary < (
  SELECT MAX(salary) 
  FROM employee
);
```
### **Senior Data Engineer Perspective**
- Global Sort Avoidance: If you solve this using ORDER BY salary DESC LIMIT 1 OFFSET 1, you are forcing the database to perform a global sort. In a distributed environment, a global sort pushes all data onto a single executor.

- Map-Reduce Efficiency: The MAX() approach is inherently a Map-Reduce operation. Each worker node independently finds its local maximum salary, and only those few numbers are sent to the driver node to compute the final maximum. This is infinitely more scalable across an AWS cluster than a global sort.

## **Medium Lesson 3: Snapchat - "Sending vs. Opening Snaps"**
### **The Problem**
Calculate the percentage of time spent sending versus opening snaps for each age group. Output the age_bucket, the send percentage, and the open percentage, rounded to 2 decimal places.
(Tables: activities [activity_id, user_id, activity_type, time_spent], age_breakdown [user_id, age_bucket])

### **The Logic (Conditional Aggregation & Math)**
- Join the fact table (activities) with the dimension table (age_breakdown).

- Sum the time spent based on the activity_type (using conditional CASE statements).

- Calculate the percentages from those grouped sums.

### **The Solution (PostgreSQL)**
``` SQL
WITH snap_stats AS (
  SELECT 
    ab.age_bucket,
    SUM(CASE WHEN a.activity_type = 'send' THEN a.time_spent ELSE 0 END) AS send_time,
    SUM(CASE WHEN a.activity_type = 'open' THEN a.time_spent ELSE 0 END) AS open_time,
    SUM(CASE WHEN a.activity_type IN ('send', 'open') THEN a.time_spent ELSE 0 END) AS total_time
  FROM activities a
  JOIN age_breakdown ab 
    ON a.user_id = ab.user_id
  GROUP BY ab.age_bucket
)

SELECT 
  age_bucket,
  ROUND(100.0 * send_time / total_time, 2) AS send_perc,
  ROUND(100.0 * open_time / total_time, 2) AS open_perc
FROM snap_stats;
```

### **Senior Data Engineer Perspective**
- Broadcast Joins: The activities table is likely billions of rows, while age_breakdown (or the distinct age buckets) is tiny. You must ensure the query optimizer executes this as a Broadcast Hash Join, replicating the small table to all nodes rather than shuffling the massive activities table.

## **Medium Lesson 4: Twitter - "Tweets' Rolling Averages"**
### **The Problem**
Given a table of tweet data over a specified time period, calculate the 3-day rolling average of tweets for each user. Output the user_id, tweet_date, and the rolling average rounded to 2 decimal places.
(Table: tweets [user_id, tweet_date, tweet_count])

### **The Logic (Window Frames)**
Use a Window Function to calculate the average. The tricky part is defining the "frame" of the window. We don't want the average of all preceding rows, just the current row and the 2 preceding rows.

### **The Solution (PostgreSQL)**
```SQL
SELECT 
  user_id, 
  tweet_date,
  ROUND(
    AVG(tweet_count) OVER (
      PARTITION BY user_id 
      ORDER BY tweet_date 
      ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2
  ) AS rolling_avg_3d
FROM tweets;
```
### **Senior Data Engineer Perspective**
- Sliding Windows in Streams: Defining frames (ROWS BETWEEN...) is the exact logic used in real-time processing frameworks like Spark Structured Streaming. When calculating rolling metrics on live data, a senior engineer also has to configure Watermarking to tell the system how long to wait for late-arriving data before finalizing that specific 3-day window calculation.

# Data Engineering Interview Prep: Medium SQL (Lessons 5 - 8)
## Topic: Ranking, Ratios, and Data Unification

---

### **Medium Lesson 5: Amazon - "Highest-Grossing Items"**

#### **The Problem**
Identify the top two highest-grossing products within each category in 2022. Output the category, product, and total spend.
*(Table: `product_spend` [category, product, user_id, spend, transaction_date])*

#### **The Logic (Two-Stage Aggregation & Ranking)**
1. First, calculate the total spend for each product within its category for the year 2022.
2. Second, use a window function (`RANK()` or `DENSE_RANK()`) to assign a rank to each product within its category based on the total spend.
3. Finally, filter the results to only include ranks 1 and 2.

#### **The Solution (PostgreSQL)**
```sql
WITH category_totals AS (
  SELECT 
    category, 
    product, 
    SUM(spend) AS total_spend
  FROM product_spend
  WHERE EXTRACT(YEAR FROM transaction_date) = 2022
  GROUP BY category, product
),
ranked_spend AS (
  SELECT 
    category, 
    product, 
    total_spend,
    RANK() OVER (
      PARTITION BY category 
      ORDER BY total_spend DESC
    ) AS rank_num
  FROM category_totals
)

SELECT category, product, total_spend
FROM ranked_spend
WHERE rank_num <= 2;
```

#### **Senior Data Engineer Perspective**
* **Ranking Functions at Scale:** In PySpark, calculating ranks requires partitioning the data by `category`. If one category (like "Electronics") has billions of rows while "Office Supplies" has a few thousand, you will hit severe **Data Skew**. In a Senior DE role, you must be prepared to handle this by using a two-pass aggregation strategy (calculating local top-N per partition before doing a global top-N per category) to avoid Out-Of-Memory errors on the executor handling the skewed partition.

---

### **Medium Lesson 6: FAANG - "Top Three Salaries"**

#### **The Problem**
Find the top 3 highest-paid employees in each department. Output the department name, employee name, and salary.
*(Tables: `employee` [employee_id, name, salary, department_id], `department` [department_id, department_name])*

#### **The Logic (Windowing with Joins)**
Join the tables to get the department names, then use `DENSE_RANK()` over the department partitioned by salary descending. `DENSE_RANK()` is preferred here because if two employees tie for the highest salary, the next highest salary is ranked #2, not #3.

#### **The Solution (PostgreSQL)**
```sql
WITH ranked_salaries AS (
  SELECT 
    d.department_name, 
    e.name, 
    e.salary,
    DENSE_RANK() OVER (
      PARTITION BY d.department_id 
      ORDER BY e.salary DESC
    ) AS salary_rank
  FROM employee e
  JOIN department d 
    ON e.department_id = d.department_id
)

SELECT department_name, name, salary
FROM ranked_salaries
WHERE salary_rank <= 3;
```

#### **Senior Data Engineer Perspective**
* **Join Before or After Windowing?** In this SQL query, we join *before* we window. In a massive data lake ecosystem (like Databricks), the `employee` table could be huge. It is often much faster to compute the `DENSE_RANK()` entirely within the massive `employee` table first, filter it down to just the top 3 IDs, and *then* join the resulting tiny dataset to the `department` table.

---

### **Medium Lesson 7: TikTok - "Signup Activation Rate"**

#### **The Problem**
Calculate the activation rate of TikTok users, rounded to 2 decimal places. The activation rate is defined as the number of users who confirmed their sign-up divided by the total number of users who attempted to sign up.
*(Tables: `emails` [email_id, user_id, signup_date], `texts` [text_id, email_id, signup_action])*

#### **The Logic (Left Joins and Decimal Division)**
1. We need all users who tried to sign up, so we start with the `emails` table.
2. We `LEFT JOIN` the `texts` table, filtering specifically for the 'Confirmed' action in the `ON` clause.
3. Divide the count of confirmed emails by the count of distinct users, casting to `DECIMAL` to prevent integer math truncation.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  ROUND(
    COUNT(t.email_id)::DECIMAL / 
    COUNT(DISTINCT e.user_id)
  , 2) AS confirm_rate
FROM emails e
LEFT JOIN texts t 
  ON e.email_id = t.email_id 
  AND t.signup_action = 'Confirmed';
```

#### **Senior Data Engineer Perspective**
* **One-to-Many Traps:** A Senior DE must immediately ask: *"Can a user have multiple confirmed text events?"* If the source system retries SMS deliveries, a single `email_id` might have multiple 'Confirmed' rows in the `texts` table. A basic `COUNT(t.email_id)` would artificially inflate the numerator. Using `COUNT(DISTINCT t.email_id)` is the safer, defensive programming approach expected at a senior level.

---

### **Medium Lesson 8: Spotify - "Spotify Streaming History"**

#### **The Problem**
You are given two tables: `songs_history` (historical play counts) and `songs_weekly` (raw streaming event logs from the current week). Write a query to output the `user_id`, `song_id`, and the cumulative count of song plays up to August 4th, 2022.
*(Tables: `songs_history` [user_id, song_id, song_plays], `songs_weekly` [user_id, song_id, listen_time])*

#### **The Logic (Union All and Unification)**
1. Aggregate the raw logs from `songs_weekly` to get the counts per user/song, filtering out events after Aug 4th.
2. Use `UNION ALL` to stack this weekly aggregated data on top of the `songs_history` data.
3. Wrap that in a CTE, and do a final `SUM()` over the combined data.

#### **The Solution (PostgreSQL)**
```sql
WITH combined_data AS (
  -- Pull historical counts
  SELECT user_id, song_id, song_plays
  FROM songs_history
  
  UNION ALL
  
  -- Calculate weekly counts up to the cutoff date
  SELECT user_id, song_id, COUNT(song_id) AS song_plays
  FROM songs_weekly
  WHERE listen_time <= '2022-08-04 23:59:59'
  GROUP BY user_id, song_id
)

SELECT 
  user_id, 
  song_id, 
  SUM(song_plays) AS total_plays
FROM combined_data
GROUP BY user_id, song_id
ORDER BY total_plays DESC;
```

#### **Senior Data Engineer Perspective**
* **Lambda Architecture Unification:** This question perfectly mimics a real-world **Lambda Architecture**. You have a "Batch Layer" (`songs_history`, likely computed overnight) and a "Speed Layer" (`songs_weekly`, arriving in near real-time). A Senior DE's job is to build queries or views that seamlessly stitch these two data stores together to give downstream analysts a unified, up-to-the-minute view of the metrics without recalculating the entire history from scratch.

# Data Engineering Interview Prep: Medium SQL (Lessons 9 - 12)
## Topic: Relational Division, Time-Series Grouping, and Conditional Logic

---

### **Medium Lesson 9: Microsoft - "Supercloud Customer"**

#### **The Problem**
A "Supercloud Customer" is a company that buys at least one product from *every* product category Microsoft offers. Write a query to return the customer IDs of these Supercloud customers.
*(Tables: `customer_contracts` [customer_id, product_id], `products` [product_id, product_category])*

#### **The Logic (Relational Division)**
1. Find the total number of unique product categories that exist in the `products` table.
2. Join the contracts to the products to see what categories each customer bought.
3. Group by the customer and count their *distinct* purchased categories.
4. If their distinct count matches the total unique categories, they are a Supercloud customer.

#### **The Solution (PostgreSQL)**
```sql
SELECT c.customer_id
FROM customer_contracts c
JOIN products p 
  ON c.product_id = p.product_id
GROUP BY c.customer_id
HAVING COUNT(DISTINCT p.product_category) = (
  SELECT COUNT(DISTINCT product_category) FROM products
);
```

#### **Senior Data Engineer Perspective**
* **Dynamic Thresholds:** Hardcoding the number of categories (e.g., `HAVING COUNT = 3`) is a junior mistake. Categories change over time. Using a subquery `(SELECT COUNT(DISTINCT...))` ensures the pipeline doesn't break when Microsoft adds a new category. 
* **Broadcast Subqueries:** In Spark, that subquery results in a single integer. The Spark Catalyst Optimizer is smart enough to evaluate that subquery once, broadcast that integer to all worker nodes, and then execute the `GROUP BY` efficiently.

---

### **Medium Lesson 10: Google - "Odd and Even Measurements"**

#### **The Problem**
Given a table of sensor measurements, calculate the sum of odd-numbered and even-numbered measurements separately for a particular day. The sequence (1st, 2nd, 3rd) is determined by the chronological order of the measurements on that specific day.
*(Table: `measurements` [measurement_id, measurement_value, measurement_time])*

#### **The Logic (Windowing by Date & Modulo Math)**
1. Extract the Date from the Timestamp.
2. Use `ROW_NUMBER()` to assign a chronological sequence (1, 2, 3...) to each measurement, partitioned by the extracted date.
3. Use modulo math (`% 2 = 0` for even, `% 2 != 0` for odd) inside conditional `SUM` statements to pivot the data.

#### **The Solution (PostgreSQL)**
```sql
WITH ranked_measurements AS (
  SELECT 
    CAST(measurement_time AS DATE) AS measurement_day, 
    measurement_value, 
    ROW_NUMBER() OVER (
      PARTITION BY CAST(measurement_time AS DATE) 
      ORDER BY measurement_time
    ) AS measurement_num 
  FROM measurements
)

SELECT 
  measurement_day, 
  SUM(CASE WHEN measurement_num % 2 != 0 THEN measurement_value ELSE 0 END) AS odd_sum,
  SUM(CASE WHEN measurement_num % 2 = 0 THEN measurement_value ELSE 0 END) AS even_sum
FROM ranked_measurements
GROUP BY measurement_day
ORDER BY measurement_day;
```

#### **Senior Data Engineer Perspective**
* **Time-Series Bucketing:** This query relies on casting a timestamp to a DATE. In large IoT pipelines, partitioning by a daily cast can still result in massive partitions. A Senior DE would often implement **minute or hour bucketing** upstream to keep the partitions manageable before doing this kind of sequential analysis.

---

### **Medium Lesson 11: Zomato - "Swapped Food Delivery"**

#### **The Problem**
Zomato wants to swap the `order_id` of consecutive orders (swap order 1 with 2, 3 with 4, etc.). If the total number of orders is odd, the last order ID remains the same.
*(Table: `orders` [order_id, item_name])*

#### **The Logic (Mathematical Swapping)**
Instead of complex joins, we can use a `CASE` statement to directly manipulate the ID.
* If the ID is odd and is *not* the max ID in the table, add 1.
* If the ID is even, subtract 1.
* If the ID is odd and *is* the max ID, leave it alone.

#### **The Solution (PostgreSQL)**
```sql
WITH max_id_table AS (
  SELECT MAX(order_id) AS max_id FROM orders
)

SELECT 
  CASE 
    WHEN order_id % 2 != 0 AND order_id != (SELECT max_id FROM max_id_table) THEN order_id + 1
    WHEN order_id % 2 = 0 THEN order_id - 1
    ELSE order_id 
  END AS corrected_order_id,
  item_name
FROM orders
ORDER BY corrected_order_id ASC;
```

#### **Senior Data Engineer Perspective**
* **Distributed Sequence Generation:** In a distributed system like Databricks, generating sequential `order_id`s without gaps is notoriously difficult and slow because it requires centralized coordination. If order IDs have gaps (e.g., 1, 2, 5, 6), this math-based approach completely breaks. A more robust (but slower) solution would involve `LEAD()` and `LAG()` window functions over chronological timestamps rather than relying on sequential IDs.

---

### **Medium Lesson 12: Bloomberg - "FAANG Stock Min-Max (Part 1)"**

#### **The Problem**
Find the highest and lowest closing prices for each FAANG stock, along with the corresponding dates. Output the ticker symbol, the highest price, the date of the highest price, the lowest price, and the date of the lowest price.
*(Table: `stock_prices` [ticker, date, close])*

#### **The Logic (Multiple Window Rankings)**
You cannot just use `MIN()` and `MAX()` because you need the *date* associated with those values. We must use `ROW_NUMBER()` twice: once sorting descending to find the highest price row, and once sorting ascending to find the lowest price row.

#### **The Solution (PostgreSQL)**
```sql
WITH ranked_prices AS (
  SELECT 
    ticker,
    date,
    close,
    ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY close DESC, date DESC) AS max_rank,
    ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY close ASC, date DESC) AS min_rank
  FROM stock_prices
)

SELECT 
  max_prices.ticker,
  max_prices.close AS highest_price,
  max_prices.date AS highest_price_date,
  min_prices.close AS lowest_price,
  min_prices.date AS lowest_price_date
FROM 
  (SELECT * FROM ranked_prices WHERE max_rank = 1) max_prices
JOIN 
  (SELECT * FROM ranked_prices WHERE min_rank = 1) min_prices
ON max_prices.ticker = min_prices.ticker;
```

#### **Senior Data Engineer Perspective**
* **Tie-Breakers:** Notice the `ORDER BY close DESC, date DESC`. Adding the date as a secondary sort is crucial. If a stock hits its all-time high on two different days, `ROW_NUMBER()` needs a deterministic tie-breaker so the pipeline doesn't return randomly fluctuating dates on different runs.

# Data Engineering Interview Prep: Medium SQL (Lessons 13 - 19)
## Topic: Consecutive Sequences, Latest Status, and Advanced Aggregations

---

### **Medium Lesson 13: Amazon - "Best-Selling Product"**

#### **The Problem**
Write a query to find the best-selling product (by highest quantity sold) for each month in 2022. Output the month, product ID, and the total quantity sold for that month.
*(Table: `sales` [transaction_id, product_id, quantity, transaction_date])*

#### **The Logic (Window Functions with Aggregation)**
1. Group by month and product to get the total quantity sold for each product per month.
2. Use `RANK()` over the month, ordered by the total quantity descending.
3. Filter the results to only keep the rank 1 products.

#### **The Solution (PostgreSQL)**
```sql
WITH monthly_sales AS (
  SELECT 
    EXTRACT(MONTH FROM transaction_date) AS month,
    product_id,
    SUM(quantity) AS total_quantity
  FROM sales
  WHERE EXTRACT(YEAR FROM transaction_date) = 2022
  GROUP BY EXTRACT(MONTH FROM transaction_date), product_id
),
ranked_sales AS (
  SELECT 
    month, 
    product_id, 
    total_quantity,
    RANK() OVER (PARTITION BY month ORDER BY total_quantity DESC) AS rnk
  FROM monthly_sales
)

SELECT month, product_id, total_quantity
FROM ranked_sales
WHERE rnk = 1;
```

#### **Senior Data Engineer Perspective**
* **Handling Ties:** `RANK()` allows ties (e.g., if two products both sold exactly 5,000 units, both get rank 1). As a Senior DE, you must confirm with stakeholders what the expected behavior is when a tie occurs. Should the dashboard show both, or should you implement a deterministic tie-breaker (like sorting by `product_id` alphabetically)?

---

### **Medium Lesson 14: Amazon - "User Shopping Sprees"**

#### **The Problem**
Identify users who have gone on a "shopping spree," defined as making purchases on 3 or more consecutive days. Output the `user_id`.
*(Table: `transactions` [transaction_id, user_id, transaction_date])*

#### **The Logic (The "Gaps and Islands" Problem)**
This is a classic "Gaps and Islands" scenario. A robust way to find consecutive days is to use `DENSE_RANK()`. If you subtract a sequential rank from a chronological sequence of dates, consecutive dates will all yield the exact same "island" date! 

#### **The Solution (PostgreSQL)**
```sql
WITH unique_days AS (
  -- Users might buy multiple things on the same day, so we need distinct days
  SELECT DISTINCT user_id, CAST(transaction_date AS DATE) AS buy_date
  FROM transactions
),
islands AS (
  SELECT 
    user_id,
    buy_date,
    buy_date - DENSE_RANK() OVER (PARTITION BY user_id ORDER BY buy_date)::INT AS island_group
  FROM unique_days
)

SELECT DISTINCT user_id
FROM islands
GROUP BY user_id, island_group
HAVING COUNT(buy_date) >= 3
ORDER BY user_id;
```

#### **Senior Data Engineer Perspective**
* **Distributed Sessionization:** In a real-time environment (like Databricks streaming), calculating consecutive days dynamically is expensive. A Senior DE would typically implement "Sessionization" upstream. When a user buys something, a separate event-driven microservice updates a "current_streak" counter in a fast NoSQL database (like DynamoDB or Redis) so the analytics team doesn't have to compute window functions across massive historical datasets.

---

### **Medium Lesson 15: Walmart - "Histogram of Users and Purchases"**

#### **The Problem**
Find the most recent transaction date for each user and count how many items they purchased on that specific day. Output a histogram of the users and their latest day's purchase count.
*(Table: `user_transactions` [transaction_id, user_id, transaction_date, product_id])*

#### **The Logic (Latest Status / First_Value)**
Use `RANK()` ordered by `transaction_date DESC` to isolate the most recent day for each user, then group by that date and count the items.

#### **The Solution (PostgreSQL)**
```sql
WITH latest_transactions AS (
  SELECT 
    user_id,
    transaction_date,
    product_id,
    RANK() OVER (PARTITION BY user_id ORDER BY transaction_date DESC) AS recent_rank
  FROM user_transactions
)

SELECT 
  transaction_date,
  user_id,
  COUNT(product_id) AS purchase_count
FROM latest_transactions
WHERE recent_rank = 1
GROUP BY transaction_date, user_id
ORDER BY transaction_date;
```

#### **Senior Data Engineer Perspective**
* **Slowly Changing Dimensions (SCD Type 2):** Finding the "latest status" is a foundational concept for Data Warehousing. While this query works perfectly, scanning an entire transaction log to find the latest date is highly inefficient. Senior engineers use SCD Type 2 tables with `is_current = TRUE` flags so that querying the "latest state" avoids window functions entirely.

---

### **Medium Lesson 16: Alibaba - "Compressed Mode"**

#### **The Problem**
Find the mode (the most frequent `item_count`) from a compressed table. Output the `item_count` in ascending order.
*(Table: `items_per_order` [item_count, order_occurrences])*

#### **The Logic (Max Subquery)**
Since the data is already pre-aggregated (compressed), the mode is simply the row(s) where `order_occurrences` is equal to the maximum `order_occurrences` in the entire table.

#### **The Solution (PostgreSQL)**
```sql
SELECT item_count
FROM items_per_order
WHERE order_occurrences = (
  SELECT MAX(order_occurrences) 
  FROM items_per_order
)
ORDER BY item_count ASC;
```

#### **Senior Data Engineer Perspective**
* **Double Scans:** This query requires scanning the table twice (once for the MAX, once for the filter). In Parquet/ORC, that's fast because the MAX is in the file metadata. But if you needed to do this grouped by a category, you would pivot to using a Window Function `MAX(order_occurrences) OVER()` to avoid a costly self-join on a massive fact table.

---

### **Medium Lesson 17: JPMorgan - "Card Launch Success"**

#### **The Problem**
Identify the "launch month" (the very first month a card was issued) and the number of cards issued in that month for each specific card name. Output the card name and the launch month issued amount.
*(Table: `monthly_cards_issued` [issue_month, issue_year, card_name, issued_amount])*

#### **The Logic (Chronological Ranking)**
We need to find the earliest chronological record for each card. We partition by `card_name` and order by `issue_year` and `issue_month` ascending.

#### **The Solution (PostgreSQL)**
```sql
WITH ranked_launches AS (
  SELECT 
    card_name,
    issued_amount,
    ROW_NUMBER() OVER (
      PARTITION BY card_name 
      ORDER BY issue_year ASC, issue_month ASC
    ) AS launch_rank
  FROM monthly_cards_issued
)

SELECT card_name, issued_amount
FROM ranked_launches
WHERE launch_rank = 1
ORDER BY issued_amount DESC;
```

#### **Senior Data Engineer Perspective**
* **Compound Sorting:** Sorting by Year then Month is correct, but dealing with separate Year and Month integers is dangerous. If a junior developer forgets the Year sort, the pipeline breaks quietly. A Senior DE would typically concatenate them into a proper `DATE` type upon ingestion so that time-series analysis is foolproof.

---

### **Medium Lesson 18: Verizon - "International Call Percentage"**

#### **The Problem**
Calculate the percentage of international calls. An international call is where the caller's country is different from the receiver's country. Round to 1 decimal place.
*(Tables: `phone_calls` [caller_id, receiver_id, call_time], `phone_info` [caller_id, country_id])*

#### **The Logic (Dual Joins to the Same Dimension)**
You must join the `phone_calls` table to the `phone_info` table *twice*: once to get the caller's country, and a second time to get the receiver's country. 

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  ROUND(
    100.0 * SUM(CASE WHEN caller.country_id != receiver.country_id THEN 1 ELSE 0 END) 
    / COUNT(*)
  , 1) AS international_calls_pct
FROM phone_calls calls
JOIN phone_info caller 
  ON calls.caller_id = caller.caller_id
JOIN phone_info receiver 
  ON calls.receiver_id = receiver.caller_id;
```

#### **Senior Data Engineer Perspective**
* **Dimension Lookups at Scale:** In a massive telecom environment, joining a multi-billion row CDR (Call Detail Record) table to a subscriber dimension table twice will trigger massive network shuffles. To optimize, ensure the `phone_info` dimension is physically broadcasted to the workers, or denormalize the country IDs directly into the fact table during streaming ingestion.

---

### **Medium Lesson 19: UnitedHealth - "Patient Support Analysis (Part 2)"**

#### **The Problem**
Find the percentage of calls that cannot be categorized (where `call_category` is either `NULL` or exactly `'n/a'`). Round the result to 1 decimal place.
*(Table: `callers` [case_id, call_category])*

#### **The Logic (Conditional Math and Null Handling)**
Count the total cases, and use a conditional `SUM` to count the uncategorized cases. 

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  ROUND(
    100.0 * SUM(CASE WHEN call_category IS NULL OR call_category = 'n/a' THEN 1 ELSE 0 END) 
    / COUNT(case_id)
  , 1) AS uncategorized_call_pct
FROM callers;
```

#### **Senior Data Engineer Perspective**
* **Standardizing Bad Data:** Handling `NULL` and string literals like `'n/a'` in analytical queries is a red flag. A Senior DE building the pipeline would implement a data quality framework (like Great Expectations) to standardize all missing or unknown categories into a single dimension key (e.g., `'Unknown'`) during the ETL phase. This prevents analysts from having to write complex `OR` conditions in every dashboard query.

# Data Engineering Interview Prep: Hard SQL (Complete 1-14)
## Topic: Retention, Hierarchies, Combinatorics, and Sequence Analytics

---

## Table of Contents
1. [Facebook: Active User Retention](#hard-lesson-1-facebook---active-user-retention)
2. [Wayfair: Y-on-Y Growth Rate](#hard-lesson-2-wayfair---y-on-y-growth-rate)
3. [Amazon: Maximize Prime Item Inventory](#hard-lesson-3-amazon---maximize-prime-item-inventory)
4. [Google: Median Google Search Frequency](#hard-lesson-4-google---median-google-search-frequency)
5. [Facebook: Advertiser Status](#hard-lesson-5-facebook---advertiser-status)
6. [Stripe: Repeated Payments](#hard-lesson-6-stripe---repeated-payments)
7. [McKinsey: 3-Topping Pizzas](#hard-lesson-7-mckinsey---3-topping-pizzas)
8. [Intuit: Consecutive Filing Years](#hard-lesson-8-intuit---consecutive-filing-years)
9. [Amazon: Server Utilization Time](#hard-lesson-9-amazon---server-utilization-time)
10. [Snowflake: Marketing Touch Streak](#hard-lesson-10-snowflake---marketing-touch-streak)
11. [UnitedHealth: Patient Support Analysis (Part 3)](#hard-lesson-11-unitedhealth---patient-support-analysis-part-3)
12. [UnitedHealth: Patient Support Analysis (Part 4)](#hard-lesson-12-unitedhealth---patient-support-analysis-part-4)
13. [Facebook: Reactivated Users](#hard-lesson-13-facebook---reactivated-users)
14. [Google: Senior Managers](#hard-lesson-14-google---senior-managers)

---

### **Hard Lesson 1: Facebook - "Active User Retention"**

#### **The Problem**
Calculate Monthly Active Users (MAUs) for July 2022. An active user performed actions in the current month (July) and the previous month (June).

#### **The Logic**
Use an `EXISTS` subquery to count distinct users in July only if they also have a record in June.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  EXTRACT(MONTH FROM curr_month.event_date) AS month, 
  COUNT(DISTINCT curr_month.user_id) AS monthly_active_users 
FROM user_actions curr_month 
WHERE EXTRACT(MONTH FROM curr_month.event_date) = 7 
  AND EXTRACT(YEAR FROM curr_month.event_date) = 2022 
  AND EXISTS (
    SELECT 1 
    FROM user_actions prev_month 
    WHERE curr_month.user_id = prev_month.user_id 
      AND EXTRACT(MONTH FROM prev_month.event_date) = 6 
      AND EXTRACT(YEAR FROM prev_month.event_date) = 2022
  )
GROUP BY EXTRACT(MONTH FROM curr_month.event_date);
```

#### **Senior Data Engineer Perspective**
A self-join (or `EXISTS`) across two months of raw event logs is an anti-pattern in production. You would use an **Accumulating Snapshot table** tracking `first_active_date`, `last_active_date`, and an array of active months to turn a multi-terabyte shuffle into a simple filter query.

---

### **Hard Lesson 2: Wayfair - "Y-on-Y Growth Rate"**

#### **The Problem**
Calculate the year-on-year growth rate for the total spend of each product.

#### **The Logic**
Aggregate spend per product per year, use the `LAG()` window function to pull the previous year's spend into the current row, and calculate the percentage change.

#### **The Solution (PostgreSQL)**
```sql
WITH yearly_spend AS (
  SELECT 
    EXTRACT(YEAR FROM transaction_date) AS year,
    product_id,
    SUM(spend) AS curr_year_spend
  FROM user_transactions
  GROUP BY EXTRACT(YEAR FROM transaction_date), product_id
),
lagged_spend AS (
  SELECT 
    year, 
    product_id, 
    curr_year_spend, 
    LAG(curr_year_spend) OVER (PARTITION BY product_id ORDER BY year ASC) AS prev_year_spend 
  FROM yearly_spend
)

SELECT 
  year,
  product_id,
  curr_year_spend,
  prev_year_spend,
  ROUND(100.0 * (curr_year_spend - prev_year_spend) / prev_year_spend, 2) AS yoy_rate
FROM lagged_spend;
```

#### **Senior Data Engineer Perspective**
If a product has zero sales in 2021 but sales in 2020 and 2022, `LAG()` will falsely compare 2022 to 2020. A Senior DE populates the base CTE with a "Date Dimension" spine to enforce `0` values for missing years before applying window functions.

---

### **Hard Lesson 3: Amazon - "Maximize Prime Item Inventory"**

#### **The Problem**
Fill a 500,000 sq ft warehouse with as many complete batches of Prime items as possible, then fill the remainder with batches of Non-Prime items.

#### **The Logic**
Calculate total sq ft for a single batch of Prime/Non-Prime items. Use Floor Division to find Prime batches, Modulo to find remaining space, and Floor Division again for Non-Prime batches.

#### **The Solution (PostgreSQL)**
```sql
WITH summary AS (
  SELECT item_type, SUM(square_footage) AS total_sqft, COUNT(item_id) AS item_count
  FROM inventory
  GROUP BY item_type
),
prime_calc AS (
  SELECT 
    item_type,
    total_sqft,
    item_count,
    FLOOR(500000 / total_sqft) * item_count AS prime_items,
    500000 % total_sqft AS remaining_space
  FROM summary
  WHERE item_type = 'prime_eligible'
)

SELECT 'prime_eligible' AS item_type, prime_items AS item_count
FROM prime_calc
UNION ALL
SELECT 'not_prime' AS item_type, FLOOR(p.remaining_space / s.total_sqft) * s.item_count AS item_count
FROM summary s
CROSS JOIN prime_calc p
WHERE s.item_type = 'not_prime';
```

#### **Senior Data Engineer Perspective**
This is an algorithmic bin-packing problem. In a modern data stack, a DE pulls summary metrics into a Python service (like an AWS Lambda) to perform procedural math, as SQL is meant for set-based operations.

---

### **Hard Lesson 4: Google - "Median Google Search Frequency"**

#### **The Problem**
Find the median searches given a compressed table (search count and number of users who made that count).

#### **The Logic**
Use `GENERATE_SERIES` to unnest the data into individual rows, then use `PERCENTILE_CONT` to calculate the median.

#### **The Solution (PostgreSQL)**
```sql
WITH expanded_searches AS (
  SELECT searches
  FROM search_frequency, GENERATE_SERIES(1, num_users)
)
SELECT ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY searches)::DECIMAL, 1) AS median
FROM expanded_searches;
```

#### **Senior Data Engineer Perspective**
`GENERATE_SERIES` (or Spark's `explode()`) inflates data massively. A Senior DE relies on specialized distributed median approximations (like Spark's `approxQuantile`) to calculate medians mathematically across distributed workers without unnesting data.

---

### **Hard Lesson 5: Facebook - "Advertiser Status"**

#### **The Problem**
Update the status of Facebook advertisers (New, Existing, Churn, Resurrect) based on their activity today vs. yesterday.

#### **The Logic**
Use a `FULL OUTER JOIN` to capture users who only exist in one table or the other, and a `CASE` statement to assign the correct transition state.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  COALESCE(a.user_id, p.user_id) AS user_id,
  CASE 
    WHEN p.spend IS NULL THEN 'CHURN'
    WHEN a.status = 'CHURN' AND p.spend IS NOT NULL THEN 'RESURRECT'
    WHEN a.status IS NULL AND p.spend IS NOT NULL THEN 'NEW'
    ELSE 'EXISTING'
  END AS new_status
FROM advertiser a
FULL OUTER JOIN daily_pay p ON a.user_id = p.user_id
ORDER BY user_id;
```

#### **Senior Data Engineer Perspective**
In architectures like Delta Lake, maintaining state is done via `MERGE INTO`. A Senior DE ensures daily snapshots are partitioned and historical states preserved (SCD Type 2) for ML models.

---

### **Hard Lesson 6: Stripe - "Repeated Payments"**

#### **The Problem**
Find payments made by the same merchant to the same customer for the exact same amount within 10 minutes of the previous transaction.

#### **The Logic**
Use `LAG()` partitioned by merchant, customer, and amount to pull the previous timestamp, then calculate the difference.

#### **The Solution (PostgreSQL)**
```sql
WITH lagged_transactions AS (
  SELECT 
    merchant_id, 
    credit_card_id, 
    amount,
    transaction_timestamp,
    LAG(transaction_timestamp) OVER (
      PARTITION BY merchant_id, credit_card_id, amount 
      ORDER BY transaction_timestamp
    ) AS prev_timestamp
  FROM transactions
)
SELECT COUNT(merchant_id) AS payment_count
FROM lagged_transactions
WHERE prev_timestamp IS NOT NULL
  AND EXTRACT(EPOCH FROM (transaction_timestamp - prev_timestamp))/60 <= 10;
```

#### **Senior Data Engineer Perspective**
Batch SQL is useless for stopping live duplicate charges. A Senior DE builds this in a streaming pipeline (Flink/Kafka Streams) holding transactions in a 10-minute stateful window to actively drop duplicates.

---

### **Hard Lesson 7: McKinsey - "3-Topping Pizzas"**

#### **The Problem**
Find all combinations of 3 different pizza toppings.

#### **The Logic**
Join the table to itself 3 times using inequality operators (`<`) to force alphabetical combinations and prevent duplicates.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  CONCAT(t1.topping_name, ',', t2.topping_name, ',', t3.topping_name) AS pizza,
  t1.topping_price + t2.topping_price + t3.topping_price AS total_cost
FROM pizza_toppings t1
JOIN pizza_toppings t2 ON t1.topping_name < t2.topping_name
JOIN pizza_toppings t3 ON t2.topping_name < t3.topping_name
ORDER BY total_cost DESC, pizza ASC;
```

#### **Senior Data Engineer Perspective**
Inequalities in `JOIN` conditions create filtered Cartesian Products. Applying this to large tables (e.g., finding "3-person friend groups" among millions of users) crashes clusters with OOM errors instantly.

---

### **Hard Lesson 8: Intuit - "Consecutive Filing Years"**

#### **The Problem**
Find users who filed taxes for 3 or more consecutive years.

#### **The Logic**
Use the Gaps and Islands approach by subtracting `ROW_NUMBER()` from the `filing_year`. Consecutive years will share the same grouping integer.

#### **The Solution (PostgreSQL)**
```sql
WITH distinct_filings AS (
  SELECT DISTINCT user_id, filing_year FROM tax_filings
),
island_groups AS (
  SELECT 
    user_id,
    filing_year,
    filing_year - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY filing_year) AS streak_group
  FROM distinct_filings
)
SELECT user_id, COUNT(filing_year) AS consecutive_years
FROM island_groups
GROUP BY user_id, streak_group
HAVING COUNT(filing_year) >= 3;
```

#### **Senior Data Engineer Perspective**
Always enforce uniqueness first (`distinct_filings`). If source data has duplicate amendments for the same year, the `ROW_NUMBER` subtraction trick breaks completely.

---

### **Hard Lesson 9: Amazon - "Server Utilization Time"**

#### **The Problem**
Calculate total uptime in full days across all servers using a log of start and stop events.

#### **The Logic**
Use `LEAD()` to peek ahead to the next 'stop' event for a specific server, calculate the difference, and sum the results.

#### **The Solution (PostgreSQL)**
```sql
WITH session_pairs AS (
  SELECT 
    server_id,
    session_status,
    status_time AS start_time,
    LEAD(status_time) OVER (PARTITION BY server_id ORDER BY status_time) AS stop_time
  FROM server_utilization
)
SELECT EXTRACT(DAY FROM SUM(stop_time - start_time)) AS total_uptime_days
FROM session_pairs
WHERE session_status = 'start';
```

#### **Senior Data Engineer Perspective**
If a server crashes and never logs a "stop", `LEAD()` returns `NULL`, breaking the `SUM()`. Wrap the `LEAD()` function in a `COALESCE(..., CURRENT_TIMESTAMP)` to safely close out active sessions.

---

### **Hard Lesson 10: Snowflake - "Marketing Touch Streak"**

#### **The Problem**
Find contacts with a marketing touch for 3+ consecutive weeks who also had a `'trial_request'`.

#### **The Logic**
Truncate to week, use `LAG()` and `LEAD()` to verify previous/next week exist, and join back to check for the trial event.

#### **The Solution (PostgreSQL)**
```sql
WITH weekly_touches AS (
  SELECT DISTINCT contact_id, DATE_TRUNC('week', event_date) AS touch_week
  FROM marketing_touches
),
streak_calc AS (
  SELECT 
    contact_id,
    LAG(touch_week) OVER (PARTITION BY contact_id ORDER BY touch_week) AS prev_week,
    touch_week,
    LEAD(touch_week) OVER (PARTITION BY contact_id ORDER BY touch_week) AS next_week
  FROM weekly_touches
)
SELECT DISTINCT c.email
FROM streak_calc s
JOIN crm_contacts c ON s.contact_id = c.contact_id
JOIN marketing_touches m ON s.contact_id = m.contact_id
WHERE s.touch_week - INTERVAL '1 week' = s.prev_week 
  AND s.touch_week + INTERVAL '1 week' = s.next_week
  AND m.event_type = 'trial_request';
```

#### **Senior Data Engineer Perspective**
The `DISTINCT` in the CTE is critical. Multiple touches in the *same* week break the `LAG()` logic. Normalize time-series data to the required grain before running sequence analytics.

---

### **Hard Lesson 11: UnitedHealth - "Patient Support Analysis (Part 3)"**

#### **The Problem**
Calculate the exact time difference between consecutive calls made by the same policyholder.

#### **The Logic**
Use `LAG()` partitioned by policyholder to pull the previous call's timestamp and subtract it from the current call's timestamp.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  policy_holder_id, 
  call_date AS current_call, 
  LAG(call_date) OVER (PARTITION BY policy_holder_id ORDER BY call_date) AS previous_call, 
  call_date - LAG(call_date) OVER (PARTITION BY policy_holder_id ORDER BY call_date) AS time_difference 
FROM callers;
```

#### **Senior Data Engineer Perspective**
PostgreSQL easily returns `INTERVAL` types, but BI tools struggle to chart them. Wrap the subtraction in `EXTRACT(EPOCH FROM ...)` to convert the interval into raw seconds for universal dashboard support.

---

### **Hard Lesson 12: UnitedHealth - "Patient Support Analysis (Part 4)"**

#### **The Problem**
Find the number of policyholders who called within exactly 7 days of their previous call.

#### **The Logic**
Wrap the `LAG()` time difference logic from Part 3 in a CTE, filter for differences <= 7 days, and count distinct users.

#### **The Solution (PostgreSQL)**
```sql
WITH call_history AS (
  SELECT 
    policy_holder_id,
    call_date - LAG(call_date) OVER (PARTITION BY policy_holder_id ORDER BY call_date) AS time_between_calls
  FROM callers
)
SELECT COUNT(DISTINCT policy_holder_id) AS frequent_callers
FROM call_history
WHERE EXTRACT(DAY FROM time_between_calls) <= 7;
```

#### **Senior Data Engineer Perspective**
Windowing by ID is efficient unless you have "super-callers" (e.g., an automated system). Monitor DAG execution for straggler tasks, which indicates data skew on a specific partition key.

---

### **Hard Lesson 13: Facebook - "Reactivated Users"**

#### **The Problem**
Count users who did not log in the previous month but logged in the current month.

#### **The Logic**
Look at current month logins and use a `NOT EXISTS` subquery to ensure no login exists for the same user in the immediately preceding month.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  EXTRACT(MONTH FROM curr_month.login_date) AS mth, 
  COUNT(DISTINCT curr_month.user_id) AS reactivated_users 
FROM user_logins AS curr_month 
WHERE NOT EXISTS ( 
  SELECT 1 
  FROM user_logins AS last_month 
  WHERE curr_month.user_id = last_month.user_id 
    AND EXTRACT(MONTH FROM last_month.login_date) = EXTRACT(MONTH FROM curr_month.login_date - INTERVAL '1 month') 
)
GROUP BY EXTRACT(MONTH FROM curr_month.login_date)
ORDER BY mth;
```

#### **Senior Data Engineer Perspective**
Subtracting `- INTERVAL '1 month'` works in Postgres but can cause edge-case errors in other engines on dates like March 31. It is safer to `DATE_TRUNC` both dates to the first of the month before interval math.

---

### **Hard Lesson 14: Google - "Senior Managers"**

#### **The Problem**
Find "senior managers" (managers who manage other managers) and count their direct manager reports.

#### **The Logic**
Use hierarchical self-joins. Join the table to itself to find the manager, then join again to find the manager's manager.

#### **The Solution (PostgreSQL)**
```sql
SELECT 
  senior_managers.manager_name, 
  COUNT(DISTINCT managers.emp_id) AS direct_reportees 
FROM employees
JOIN employees AS managers ON employees.manager_id = managers.emp_id 
JOIN employees AS senior_managers ON managers.manager_id = senior_managers.emp_id 
GROUP BY senior_managers.manager_name 
ORDER BY direct_reportees DESC;
```

#### **Senior Data Engineer Perspective**
Recursive structures are ugly in relational databases due to explicit self-joins for every depth level. At scale, a Senior DE architects HR systems using Graph Databases (Neo4j/Amazon Neptune) or recursive CTEs.